# SSVEP quick-look

Offline sanity check of one recording from `src/stimulus/ssvep_session.py`.
Not a decoder: it checks whether the signal is there.

1. Load raw data, events and metadata; cross-check markers against `events.csv`
2. Eyes-open vs eyes-closed PSD (Phase 3 gate 1: alpha peak)
3. PSD per target over O1/Oz/O2 (gate 2: peaks at stimulus frequencies)
4. Plain CCA accuracy + confusion matrix (gate 3: above chance)
5. Frame-timing summary

Look at the PSDs before trusting the CCA number: the PSD does not depend on the decoder
(decision log 2026-10-22). On the **synthetic board** there is no SSVEP, so CCA should be near chance (25%).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from brainflow.data_filter import DataFilter
from scipy import signal, stats

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
from src.stimulus import markers as M

# Set to a folder name in data/raw/, or None for the most recent session.
SESSION = None
LATENCY_S = 0.0           # skip this much after flicker onset (visual latency ~0.1-0.15 s)
CCA_HARMONICS = 2
CCA_CHANNELS = None       # None = all montage channels, or e.g. ["O1", "Oz", "O2"]
PSD_CHANNELS = ["O1", "Oz", "O2"]
LINE_HZ = 60.0
MAX_DROPPED_FRAMES = None  # e.g. 2: exclude flicker blocks with more dropped frames than this

In [ ]:
raw_root = REPO / "data" / "raw"
if SESSION is None:
    candidates = sorted(p for p in raw_root.iterdir() if (p / "events.csv").exists())
    session_dir = max(candidates, key=lambda p: p.stat().st_mtime)
else:
    session_dir = raw_root / SESSION
print("session:", session_dir.name)

meta = json.loads((session_dir / "metadata.json").read_text())
events = pd.read_csv(session_dir / "events.csv")
data = DataFilter.read_file(str(session_dir / "raw.csv"))   # rows x samples

board = meta["board"]
fs = board["sampling_rate_hz"]
montage = list(board["montage"].keys())
eeg = data[board["eeg_channel_rows"]]
marker_row = data[board["marker_channel_row"]]
ts = data[board["timestamp_channel_row"]]
targets = meta["stimulus"]["targets"]   # name -> {marker_k, frames_per_cycle, ...}
k_to_name = {v["marker_k"]: n for n, v in targets.items()}
k_to_freq = {v["marker_k"]: v["freq_hz_at_measured_refresh"] for v in targets.values()}

print(f"board: {board['board_name']}, fs = {fs} Hz, {eeg.shape[1]} samples ({eeg.shape[1] / fs:.1f} s)")
print(f"effective fs from timestamps: {(len(ts) - 1) / (ts[-1] - ts[0]):.2f} Hz")
print("aborted:", meta["aborted"], "| trials completed:", meta["stimulus"]["trials_completed"])
print("stimulus freqs (at measured refresh):", {k_to_name[k]: round(f, 3) for k, f in k_to_freq.items()})
if board["board_name"] == "synthetic":
    print("NOTE: synthetic board -> no real EEG; expect no alpha/SSVEP peaks and chance-level CCA.")

## Marker check

Every event in `events.csv` should appear in the marker channel in the same order. Intervals between
markers (in EEG samples) are compared with intervals between PsychoPy flips. This measures host-side
consistency only, **not** true stimulus-to-EEG latency (Bluetooth latency is not visible here).

In [ ]:
idx = np.flatnonzero(marker_row)
raw_codes = marker_row[idx].astype(int)
csv_codes = events["code"].to_numpy()
print(f"{len(raw_codes)} markers in raw, {len(csv_codes)} in events.csv")
assert np.array_equal(raw_codes, csv_codes), "marker sequence mismatch between raw data and events.csv"

ev = events.assign(sample=idx)
flip = ev["psychopy_flip_s"].to_numpy()
ok = ~np.isin(ev["code"], [M.ABORT, M.SESSION_END])  # these are not inserted on a flip
err_ms = (np.diff(idx[ok]) / fs - np.diff(flip[ok])) * 1000
print(f"marker interval (EEG samples) minus flip interval: "
      f"mean {err_ms.mean():.2f} ms, sd {err_ms.std():.2f} ms, max |err| {np.abs(err_ms).max():.2f} ms "
      f"(1 sample = {1000 / fs:.1f} ms)")

## Filtering

Notch at line frequency, then 3-40 Hz band-pass, zero-phase, on the continuous recording.

In [ ]:
b, a = signal.iirnotch(LINE_HZ, Q=30, fs=fs)
x = signal.filtfilt(b, a, eeg, axis=1)
sos = signal.butter(4, [3, 40], btype="bandpass", fs=fs, output="sos")
x = signal.sosfiltfilt(sos, x, axis=1)
ch = {name: i for i, name in enumerate(montage)}
psd_idx = [ch[c] for c in PSD_CHANNELS]

## Eyes open vs eyes closed (alpha check)

In [ ]:
def segment(start_code, end_code):
    s = ev.loc[ev.code == start_code, "sample"]
    e = ev.loc[ev.code == end_code, "sample"]
    if s.empty or e.empty:
        return None
    return x[:, s.iloc[0]:e.iloc[0]]

fig, axes = plt.subplots(1, len(PSD_CHANNELS), figsize=(13, 3.5), sharey=True)
alpha_ratio = {}
for label, (c0, c1), color in [("eyes open", (M.EYES_OPEN_START, M.EYES_OPEN_END), "C0"),
                               ("eyes closed", (M.EYES_CLOSED_START, M.EYES_CLOSED_END), "C1")]:
    seg = segment(c0, c1)
    if seg is None:
        print("missing", label)
        continue
    f, p = signal.welch(seg, fs=fs, nperseg=min(seg.shape[1], 4 * fs))
    alpha_ratio[label] = p[:, (f >= 8) & (f <= 13)].mean(axis=1)
    for ax, i in zip(axes, psd_idx):
        ax.semilogy(f, p[i], color=color, label=label)
for ax, name in zip(axes, PSD_CHANNELS):
    ax.axvspan(8, 13, color="0.9", zorder=0)
    ax.set(title=name, xlim=(2, 40), xlabel="Hz")
axes[0].set_ylabel("PSD (uV^2/Hz)")
axes[0].legend()
plt.tight_layout()
plt.show()

if len(alpha_ratio) == 2:
    r = alpha_ratio["eyes closed"] / alpha_ratio["eyes open"]
    print("alpha (8-13 Hz) power ratio closed/open:", {c: round(float(r[ch[c]]), 2) for c in PSD_CHANNELS})

## Epochs and PSD per target

In [ ]:
flicker_s = meta["stimulus"]["flicker_s"]
n_samp = int(round((flicker_s - LATENCY_S) * fs))
# Only completed trials: a flicker onset must be followed by its flicker offset (drops an aborted trial).
on_mask = (ev.code > M.FLICKER_ON_BASE) & (ev.code < M.FLICKER_ON_BASE + 100)
offs = ev[(ev.code > M.FLICKER_OFF_BASE) & (ev.code < M.FLICKER_OFF_BASE + 100)]
if MAX_DROPPED_FRAMES is not None:
    print("excluded for dropped frames:", int((offs.dropped_frames > MAX_DROPPED_FRAMES).sum()), "trials")
    offs = offs[offs.dropped_frames <= MAX_DROPPED_FRAMES]
off_trials = set(offs.trial)
onsets = ev[on_mask & ev.trial.isin(off_trials)]
starts = onsets["sample"].to_numpy() + int(round(LATENCY_S * fs))
keep = starts + n_samp <= x.shape[1]
epochs = np.stack([x[:, s:s + n_samp] for s in starts[keep]])     # trials x ch x samples
y_true = (onsets["code"].to_numpy() - M.FLICKER_ON_BASE)[keep]
print(f"{len(epochs)} epochs of {n_samp / fs:.2f} s; per target:",
      {k_to_name[k]: int((y_true == k).sum()) for k in sorted(k_to_name)})

ks = sorted(k_to_name)
fig, axes = plt.subplots(1, len(ks), figsize=(14, 3.5), sharey=True)
for ax, k in zip(axes, ks):
    ep = epochs[y_true == k][:, psd_idx]
    if len(ep) == 0:
        continue
    f, p = signal.welch(ep, fs=fs, nperseg=n_samp, axis=-1)
    ax.semilogy(f, p.mean(axis=(0, 1)), color="k")
    for other in ks:
        ax.axvline(k_to_freq[other], color="0.85", lw=1, zorder=0)
    for h in (1, 2, 3):
        ax.axvline(h * k_to_freq[k], color="C3", ls="--", lw=1)
    ax.set(title=f"{k_to_name[k]} ({k_to_freq[k]:.2f} Hz), n={len(ep)}", xlim=(3, 40), xlabel="Hz")
axes[0].set_ylabel(f"PSD, mean of {'/'.join(PSD_CHANNELS)}")
plt.tight_layout()
plt.show()

## Plain CCA

For each epoch, the canonical correlation between the EEG and sine/cosine references at each stimulus
frequency (plus harmonics). The predicted target is the frequency with the highest correlation (Lin et al. 2006).
No training, no fitted parameters.

In [ ]:
def cca_max_corr(X, Y):
    # X: samples x channels, Y: samples x refs. Largest canonical correlation via QR + SVD.
    qx, _ = np.linalg.qr(X - X.mean(0))
    qy, _ = np.linalg.qr(Y - Y.mean(0))
    return np.linalg.svd(qx.T @ qy, compute_uv=False)[0]

t = np.arange(n_samp) / fs + LATENCY_S
refs = {k: np.column_stack([fn(2 * np.pi * h * k_to_freq[k] * t)
                            for h in range(1, CCA_HARMONICS + 1) for fn in (np.sin, np.cos)])
        for k in ks}
cca_idx = [ch[c] for c in (CCA_CHANNELS or montage)]
rho = np.array([[cca_max_corr(ep[cca_idx].T, refs[k]) for k in ks] for ep in epochs])
y_pred = np.array(ks)[rho.argmax(axis=1)]

n_correct = int((y_pred == y_true).sum())
acc = n_correct / len(y_true)
chance = 1 / len(ks)
p = stats.binomtest(n_correct, len(y_true), chance, alternative="greater").pvalue
print(f"CCA accuracy: {acc:.1%} ({n_correct}/{len(y_true)}), chance {chance:.0%}, binomial p = {p:.3g}")

cm = np.zeros((len(ks), len(ks)), int)
for t_, p_ in zip(y_true, y_pred):
    cm[ks.index(t_), ks.index(p_)] += 1
names = [f"{k_to_name[k]}\n{k_to_freq[k]:.2f} Hz" for k in ks]
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(len(ks)):
    for j in range(len(ks)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set(xticks=range(len(ks)), yticks=range(len(ks)), xticklabels=names, yticklabels=names,
       xlabel="predicted", ylabel="true", title=f"CCA, {acc:.0%} correct")
plt.tight_layout()
plt.show()

## Frame timing

In [ ]:
print(json.dumps(meta["display"], indent=1))
print(json.dumps(meta["frame_timing"], indent=1))
drops = events.loc[events.label.str.startswith("flicker_off"), ["trial", "target", "dropped_frames"]]
print(f"flicker blocks with dropped frames: {(drops.dropped_frames > 0).sum()} / {len(drops)}")
drops[drops.dropped_frames > 0]